# 3. EI, PI and UCB
An acquisition scores **where to evaluate next**; it does not replace the real experiment. This independent lesson computes three maximization criteria from a GP posterior:

- EI: expected improvement over the best observed score.
- PI: probability of improving by at least `xi`.
- UCB: predicted mean plus an uncertainty bonus `beta * std`.

OpenBO's PI/UCB package modules are placeholders. Their formulas below are self-contained teaching examples; the EI helper is implemented in the package.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from openbo.models.gp_scratch import GPScratch
from openbo.acquisition.ei import expected_improvement_maximization

x_train = np.array([[0.05], [0.4], [0.85]])
y_train = -(x_train[:, 0] - 0.35)**2
gp = GPScratch(lengthscale=0.2, kernel_type="rbf")
gp.fit(x_train, y_train)
x = np.linspace(0, 1, 201)[:, None]
mean, variance = gp.posterior(x)
std = np.sqrt(np.maximum(variance, 1e-12))
best_y = y_train.max()
xi, beta = 0.0, 2.0
scores = {
    "EI": expected_improvement_maximization(mean, variance, best_y),
    "PI": norm.cdf((mean - best_y - xi) / std),
    "UCB": mean + beta * std,
}
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, (name, values) in zip(axes, scores.items()):
    chosen = int(np.argmax(values))
    ax.plot(x[:, 0], values)
    ax.axvline(x[chosen, 0], color="black", linestyle="--")
    ax.set(title=name, xlabel="Normalized input", ylabel="Acquisition score")
    print(name, "suggests", float(x[chosen, 0]))
fig.tight_layout()
plt.show()
plt.close(fig)

## Try it
Increase `xi` or `beta`. How does the next experiment move? The acquisition values have different scales, so compare their selected inputs rather than their raw magnitudes. Here selection is over a finite grid; the main optimizer also refines promising candidates continuously.

Next: [BoTorch](04_botorch_bo.ipynb), or read [the EI implementation](../src/openbo/acquisition/ei.py).